# Content-Based Filtering

## Learning Objectives

1. **Build** item profiles from TF-IDF features
2. **Construct** user profiles from rated items
3. **Generate** recommendations via cosine similarity
4. **Discuss** limitations and when content-based is preferred

## Item Profiles

Represent each item $i$ as a feature vector $\vec{f}_i \in \mathbb{R}^d$.

**For text items (movies, articles):**
Use **TF-IDF** weights for each keyword/feature:
$$\text{TF-IDF}(t, i) = \text{tf}(t, i) \times \log\frac{N}{\text{df}(t)}$$

where $\text{tf}(t,i)$ = frequency of term $t$ in item $i$, $N$ = total items, $\text{df}(t)$ = items containing term $t$.

**For other domains:**
- Movies: genre, director, cast, release year
- Products: category, brand, specifications
- Songs: tempo, key, energy (Spotify audio features)

In [1]:
import math
from collections import Counter

def tfidf_profiles(items_keywords):
    """Compute TF-IDF item profiles.
    items_keywords: {item_id: [keyword, ...]}
    """
    N = len(items_keywords)
    # Document frequency per keyword
    df = Counter()
    for kws in items_keywords.values():
        for kw in set(kws): df[kw] += 1

    vocab = sorted(df.keys())
    vocab_idx = {kw: i for i,kw in enumerate(vocab)}

    profiles = {}
    for item_id, kws in items_keywords.items():
        tf = Counter(kws)
        vec = [0.0]*len(vocab)
        for kw, count in tf.items():
            tfidf = (count/len(kws)) * math.log(N/df[kw])
            vec[vocab_idx[kw]] = tfidf
        # L2 normalize
        norm = math.sqrt(sum(v**2 for v in vec))
        profiles[item_id] = [v/norm for v in vec] if norm > 0 else vec
    return profiles, vocab

# Movie catalog
movies = {
    'M1': ['action','thriller','chase','hero'],
    'M2': ['romance','drama','love','paris'],
    'M3': ['action','superhero','hero','fight'],
    'M4': ['documentary','nature','wildlife'],
    'M5': ['romance','comedy','love','funny'],
    'M6': ['thriller','mystery','detective'],
}
profiles, vocab = tfidf_profiles(movies)
print(f"""Vocabulary ({len(vocab)} terms): {vocab}
""")
for m in ['M1','M2','M3']:
    top_terms = sorted(enumerate(profiles[m]), key=lambda x:-x[1])[:3]
    print(f"{m}: top terms = {[(vocab[i], round(v,3)) for i,v in top_terms]}")

Vocabulary (17 terms): ['action', 'chase', 'comedy', 'detective', 'documentary', 'drama', 'fight', 'funny', 'hero', 'love', 'mystery', 'nature', 'paris', 'romance', 'superhero', 'thriller', 'wildlife']

M1: top terms = [('chase', 0.686), ('action', 0.42), ('hero', 0.42)]
M2: top terms = [('drama', 0.603), ('paris', 0.603), ('love', 0.37)]
M3: top terms = [('fight', 0.603), ('superhero', 0.603), ('action', 0.37)]


In [2]:
def cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x**2 for x in a)); nb = math.sqrt(sum(x**2 for x in b))
    return dot/(na*nb) if na*nb > 0 else 0.0

def user_profile(rated_items, profiles, ratings):
    """Average TF-IDF profile weighted by user ratings."""
    d = len(next(iter(profiles.values())))
    user_vec = [0.0]*d
    total_weight = 0.0
    for item, rating in zip(rated_items, ratings):
        w = rating - 3.0  # center around neutral
        if w != 0:
            for i in range(d): user_vec[i] += w * profiles[item][i]
        total_weight += abs(w)
    if total_weight > 0:
        user_vec = [v/total_weight for v in user_vec]
    norm = math.sqrt(sum(v**2 for v in user_vec))
    return [v/norm for v in user_vec] if norm > 0 else user_vec

# User who likes action/thrillers
alice_rated = ['M1', 'M6', 'M2']
alice_ratings = [5, 4, 2]
alice_profile = user_profile(alice_rated, profiles, alice_ratings)

# Recommend unrated items
unrated = [m for m in movies if m not in alice_rated]
scores = [(m, cosine(alice_profile, profiles[m])) for m in unrated]
scores.sort(key=lambda x: -x[1])

print("Alice's top recommendations:")
for m, score in scores:
    print(f"  {m}: {score:.4f}  (keywords: {movies[m]})")

Alice's top recommendations:
  M3: 0.2406  (keywords: ['action', 'superhero', 'hero', 'fight'])
  M4: 0.0000  (keywords: ['documentary', 'nature', 'wildlife'])
  M5: -0.1058  (keywords: ['romance', 'comedy', 'love', 'funny'])


## Strengths and Limitations

**Strengths:**
- No cold start for new items (just need content features)
- Explainable: "recommended because you liked X which also has feature Y"
- Works for users with unique tastes (no need for similar users)

**Limitations:**
- Requires item features (hard for some domains)
- Filter bubble: only recommends "more of the same"
- Can't discover cross-genre preferences
- Quality depends on feature engineering